[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/20_weight_init.ipynb)

# 🟢 Easy: Kaiming Initialization

Implement **Kaiming (He) normal initialization** for weight tensors.

$$W \sim \mathcal{N}(0, \text{std}^2) \quad \text{where} \quad \text{std} = \sqrt{\frac{2}{\text{fan\_in}}}$$

### Signature
```python
def kaiming_init(weight: Tensor) -> Tensor:
    # Initialize weight in-place with Kaiming normal
    # fan_in = weight.shape[1]
    # Returns the weight tensor
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.1 MB/s eta 0:00:00


In [2]:
import torch
import math

In [28]:
# ✏️ YOUR IMPLEMENTATION HERE

#kaiming init is needed in deep NN where variance from input to output of a layer can be very different and cause vanishing/exploding gradients.
#We want the variance to be constant: since we know that sample features are from normal distribution (hypotesis or normalization)
#!we have out = W @ X --> var(out) = var(W)var(X) + var(W)E(x)^2 + var(X)E(W)^2 but E(x) = E(w) = 0 for each Xi, Wj (normal distrib.)
#!so var(out) = var(W)var(X), and since OUTk = sum(WiXi) with i from 1 to d_in then var(OUTi) = d_in * var(Wi) * var(Xi)
#so var(out) NOT constant
#we have to fix var(Wi) = 1 / d_in so the var stays constant
#!to do so we fix each Wi as 1/sqrt(d_in) * normal_w --> so var(Wi) = 1 / d_in
#!this is XAVIER init, which works fine without ReLU act
#with relu, half of the outp0ut id truncated, so its variance is the half and we'd obtain:
#!var(OUTk) = var(Xi) / 2 since var(OUTi) = d_in * var(Wi) * var(act(Xi)) and var(act(Xi)) = var(Xi) / 2
#!so we have to double var(Wi) too --> init as sqrt(2 / d_in) * normal_w --> var(Wi) = 2 / d_in and var(OUTi) = var(act(Xi))!
def kaiming_init(weight):
    d_in = weight.shape[1]
    weight[:] = torch.normal(0.0, math.sqrt(2 / d_in), weight.shape)
    return weight

In [29]:
# 🧪 Debug
import math
w = torch.empty(256, 512)
kaiming_init(w)
print(f'Mean: {w.mean():.4f} (expect ~0)')
print(f'Std:  {w.std():.4f} (expect {math.sqrt(2/512):.4f})')

Mean: -0.0002 (expect ~0)
Std:  0.0627 (expect 0.0625)


In [30]:
# ✅ SUBMIT
from torch_judge import check
check('weight_init')


🧪 Testing: Kaiming Initialization (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Mean approximately 0 (5.5ms)
  ✅ [2/4] Std matches sqrt(2/fan_in) (4.8ms)
  ✅ [3/4] Returns same tensor (in-place) (0.3ms)
  ✅ [4/4] Smaller fan_in gives larger std (0.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (11.0ms total)
  Progress saved. Run status() to see your dashboard.

